# Phase 3C — Frozen Decision Tree final-test evaluation

This notebook performs one locked final evaluation of the already-selected Decision Tree. It does not compare candidates or tune anything. Final-test results are evidence for this educational portfolio project only: they are not production, lending, or regulatory evidence and must not be used for further model selection or tuning.

## 1. Mount Google Drive

**What:** connect the Colab runtime to Drive. **Why:** the audited labeled CSV remains outside Git. **Expected:** a successful mount message. **Check:** confirm `/content/drive` is available before continuing.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Freeze configuration

**What:** define the one approved dataset path, repository branch, checksum, class order, and seed. **Why:** visible constants make the final run auditable. **Expected:** a configuration summary without customer data. **Check:** the path points to labeled `train.csv`; Kaggle `test.csv` is never requested.

In [ ]:
from pathlib import Path

DATASET_PATH = Path('/content/drive/MyDrive/Credit-Scoring-Model/data/raw/kaggle_credit_score/train.csv')
EXPECTED_SHA256 = 'D2EBCC056A64C48710B1AEB96777155835D372D7AD202529F64666011D214DA0'
REPOSITORY_URL = 'https://github.com/MaryamCodeHub/Credit-Scoring-Model.git'
REPOSITORY_DIR = Path('/content/Credit-Scoring-Model')
BRANCH = 'model-improvement-v2'
RANDOM_STATE = 42
CLASS_ORDER = ['Poor', 'Standard', 'Good']

print(f'Labeled dataset: {DATASET_PATH}')
print(f'Repository branch: {BRANCH}')
print(f'Fixed class order: {CLASS_ORDER}')

## 3. Verify the audited file before reading it

**What:** verify existence and SHA-256 using file bytes only. **Why:** the final result is meaningful only for the audited dataset. **Expected:** `Dataset file and SHA-256 verified.` **Check:** stop if this cell raises an error.

In [ ]:
import hashlib

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Labeled train.csv was not found at {DATASET_PATH}.')

digest = hashlib.sha256()
with DATASET_PATH.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
actual_sha256 = digest.hexdigest().upper()
if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(f'Wrong train.csv: expected {EXPECTED_SHA256}, got {actual_sha256}.')
print('Dataset file and SHA-256 verified.')

## 4. Clone or safely update the reviewed repository

**What:** clone the repository or fast-forward an existing Colab clone, then explicitly check out `model-improvement-v2`. **Why:** evaluation must use reviewed code. **Expected:** the active branch and commit. **Check:** the branch must exactly match the frozen configuration.

In [ ]:
import subprocess

def run_command(arguments, cwd=None):
    return subprocess.run(arguments, cwd=cwd, check=True, text=True, capture_output=True)

if (REPOSITORY_DIR / '.git').is_dir():
    run_command(['git', 'fetch', 'origin', BRANCH], cwd=REPOSITORY_DIR)
    run_command(['git', 'checkout', BRANCH], cwd=REPOSITORY_DIR)
    run_command(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPOSITORY_DIR)
else:
    run_command(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY_URL, str(REPOSITORY_DIR)])

active_branch = run_command(['git', 'branch', '--show-current'], cwd=REPOSITORY_DIR).stdout.strip()
commit_hash = run_command(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_DIR).stdout.strip()
if active_branch != BRANCH:
    raise RuntimeError(f'Wrong branch: expected {BRANCH}, found {active_branch}.')
print(f'Active branch: {active_branch}')
print(f'Commit: {commit_hash}')

## 5. Install one consistent scientific stack

**What:** inspect package metadata before importing NumPy, pandas, SciPy, or scikit-learn. If any pinned version differs, reinstall the complete binary stack and restart the Colab process. **Why:** mixing already-imported and newly installed binary packages can cause NumPy/SciPy import errors. **Expected:** either `Dependency versions already match` or an automatic restart. **Check:** after a restart, rerun the notebook from the top; this cell is idempotent and should then skip installation.

In [ ]:
import importlib.metadata as metadata
import os
import signal
import subprocess
import sys

REQUIRED_VERSIONS = {
    'numpy': '2.4.4',
    'pandas': '3.0.2',
    'scipy': '1.17.1',
    'scikit-learn': '1.8.0',
    'matplotlib': '3.10.8',
}
installed = {}
for package_name in REQUIRED_VERSIONS:
    try:
        installed[package_name] = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        installed[package_name] = None

versions_changed = any(installed[name] != version for name, version in REQUIRED_VERSIONS.items())
if versions_changed:
    specifications = [f'{name}=={version}' for name, version in REQUIRED_VERSIONS.items()]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall', *specifications])
    print('Dependencies changed. Colab will restart now; rerun every cell from the top.')
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('Dependency versions already match; no install or restart is needed.')

## 6. Import the approved workflow

**What:** import the repository splitter, cleaner, approved preprocessor, the single frozen classifier, and evaluation metrics. **Why:** reviewed functions must be reused instead of duplicating logic. **Expected:** package versions and an import confirmation. **Check:** no Random Forest, Logistic Regression, grid search, or randomized search is imported.

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

if str(REPOSITORY_DIR) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR))
from src.data_cleaning import clean_credit_data
from src.data_splitting import split_by_customer
from src.model_preprocessing import build_model_preprocessor, separate_features_target_groups

print(f'NumPy {np.__version__}; pandas {pd.__version__}; scikit-learn {sklearn.__version__}')
print('Approved evaluation imports completed.')

## 7. Load labeled data and recreate the locked customer split

**What:** read audited `train.csv` and recreate the seed-42 customer-grouped split. **Why:** the same customer must never cross partitions. **Expected:** row/customer counts and overlap confirmation only. **Check:** no names, SSNs, IDs, Customer_ID values, or raw rows are displayed, and final-test feature values are not inspected.

In [ ]:
raw_train = pd.read_csv(DATASET_PATH, low_memory=False)
partitions = split_by_customer(raw_train, random_state=RANDOM_STATE)

development_customers = set(partitions.development_train['Customer_ID'])
validation_customers = set(partitions.validation['Customer_ID'])
final_customers = set(partitions.final_test['Customer_ID'])
assert development_customers.isdisjoint(validation_customers)
assert development_customers.isdisjoint(final_customers)
assert validation_customers.isdisjoint(final_customers)

partition_summary = pd.DataFrame([
    {'Partition': 'Development', 'Rows': len(partitions.development_train), 'Customers': len(development_customers)},
    {'Partition': 'Validation', 'Rows': len(partitions.validation), 'Customers': len(validation_customers)},
    {'Partition': 'Final test (sealed)', 'Rows': len(partitions.final_test), 'Customers': len(final_customers)},
])
display(partition_summary)
print('Customer overlap is zero. Final test remains sealed.')
del development_customers, validation_customers, final_customers

## 8. Freeze the decision before final-test access

**What:** display the selected Decision Tree parameters, pre-final evidence, and acceptance criteria. **Why:** every decision must be fixed before final test is touched. A parameter is learned during fitting; a hyperparameter is chosen beforehand. These hyperparameters will not change. **Expected:** three small tables. **Check:** the values exactly match the approved decision.

The criteria below are educational portfolio criteria—not real lending, safety, fairness, regulatory, or production thresholds.

In [ ]:
FROZEN_MODEL_PARAMETERS = {
    'max_depth': 6,
    'min_samples_leaf': 100,
    'class_weight': None,
    'random_state': 42,
}
FROZEN_PRE_FINAL_EVIDENCE = {
    'CV Macro F1': 0.6170008543287092,
    'CV Std Macro F1': 0.009939724646880005,
    'Validation Macro F1': 0.619834,
    'Validation Poor recall': 0.550676,
    'Train-validation gap': 0.010378,
}
FROZEN_ACCEPTANCE_CRITERIA = {
    'Final-test Macro F1 minimum': 0.58,
    'Final-test Poor recall minimum': 0.50,
    'Every class F1 minimum': 0.45,
    'Maximum absolute validation-to-final Macro F1 difference': 0.04,
}
display(pd.DataFrame([FROZEN_MODEL_PARAMETERS], index=['Frozen Decision Tree']))
display(pd.DataFrame.from_dict(FROZEN_PRE_FINAL_EVIDENCE, orient='index', columns=['Frozen value']))
display(pd.DataFrame.from_dict(FROZEN_ACCEPTANCE_CRITERIA, orient='index', columns=['Frozen value']))
print('Model, evidence, features, and acceptance criteria are frozen before final-test access.')

## 9. Prepare combined development and validation data

**What:** clean development and validation independently, combine them, and separate approved features and target. **Why:** model selection is finished, so both pre-final partitions may train the one frozen model. **Expected:** combined row and feature counts only. **Check:** final test is not referenced or cleaned here.

In [ ]:
clean_development = clean_credit_data(partitions.development_train)
clean_validation = clean_credit_data(partitions.validation)
combined_pre_final = pd.concat([clean_development, clean_validation], axis=0).sort_index()
pre_final_inputs = separate_features_target_groups(combined_pre_final)
X_pre_final = pre_final_inputs.X
y_pre_final = pre_final_inputs.y

assert len(X_pre_final) == len(y_pre_final) == len(combined_pre_final)
print(f'Combined pre-final rows: {len(X_pre_final):,}')
print(f'Approved raw feature count: {X_pre_final.shape[1]}')
print('Development and validation are ready; final test is still sealed.')

## 10. Build and fit the one frozen pipeline

**What:** create a fresh approved preprocessing-plus-Decision-Tree pipeline and fit it once on combined development and validation. **Why:** preprocessing statistics must be learned only from the complete pre-final training data, before final test is opened. **Expected:** training time and a frozen-pipeline confirmation. **Check:** parameters exactly match the displayed decision and no other estimator is trained.

In [ ]:
frozen_pipeline = Pipeline([
    ('preprocessing', build_model_preprocessor()),
    ('classifier', DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=100,
        class_weight=None,
        random_state=42,
    )),
])
for parameter_name, frozen_value in FROZEN_MODEL_PARAMETERS.items():
    assert frozen_pipeline.named_steps['classifier'].get_params()[parameter_name] == frozen_value

training_started = time.perf_counter()
frozen_pipeline.fit(X_pre_final, y_pre_final)
training_seconds = time.perf_counter() - training_started
print(f'Frozen pipeline fitted once in {training_seconds:.2f} seconds.')
print('All fitting is complete. No parameter, feature, or threshold may change now.')

## 11. Open final test once and make the single prediction

**What:** now—and only now—clean final test for the first time, separate approved features and target, and make one prediction. **Why:** this is the locked estimate for previously unseen customers. **Expected:** final-test row count and prediction time only. **Check:** there is no fitting, tuning, candidate comparison, or raw-record display in this or any later cell.

In [ ]:
clean_final_test = clean_credit_data(partitions.final_test)
final_test_inputs = separate_features_target_groups(clean_final_test)
X_final_test = final_test_inputs.X
y_final_test = final_test_inputs.y

prediction_started = time.perf_counter()
final_test_predictions = frozen_pipeline.predict(X_final_test)
prediction_seconds = time.perf_counter() - prediction_started
assert len(final_test_predictions) == len(y_final_test)
print(f'Final-test rows evaluated once: {len(y_final_test):,}')
print(f'Prediction time: {prediction_seconds:.3f} seconds')

## 12. Calculate the locked final-test metrics once

**What:** calculate the predefined overall and per-class metrics in the fixed Poor–Standard–Good order. **Why:** Macro F1 alone can hide a weak class. **Expected:** one summary table and one per-class table. **Check:** Poor recall, every class F1, timing, and the validation-to-final difference are present.

In [ ]:
final_macro_f1 = f1_score(y_final_test, final_test_predictions, labels=CLASS_ORDER, average='macro', zero_division=0)
final_accuracy = accuracy_score(y_final_test, final_test_predictions)
final_balanced_accuracy = balanced_accuracy_score(y_final_test, final_test_predictions)
final_weighted_f1 = f1_score(y_final_test, final_test_predictions, labels=CLASS_ORDER, average='weighted', zero_division=0)
final_macro_precision = precision_score(y_final_test, final_test_predictions, labels=CLASS_ORDER, average='macro', zero_division=0)
final_macro_recall = recall_score(y_final_test, final_test_predictions, labels=CLASS_ORDER, average='macro', zero_division=0)
final_report = classification_report(y_final_test, final_test_predictions, labels=CLASS_ORDER, target_names=CLASS_ORDER, output_dict=True, zero_division=0)
final_matrix = confusion_matrix(y_final_test, final_test_predictions, labels=CLASS_ORDER)
poor_recall = final_report['Poor']['recall']
validation_macro_f1 = FROZEN_PRE_FINAL_EVIDENCE['Validation Macro F1']
signed_validation_difference = final_macro_f1 - validation_macro_f1
absolute_validation_difference = abs(signed_validation_difference)

final_summary = pd.DataFrame([{
    'Final-test Macro F1': final_macro_f1,
    'Accuracy': final_accuracy,
    'Balanced accuracy': final_balanced_accuracy,
    'Weighted F1': final_weighted_f1,
    'Macro precision': final_macro_precision,
    'Macro recall': final_macro_recall,
    'Poor recall': poor_recall,
    'Final minus validation Macro F1': signed_validation_difference,
    'Absolute validation-to-final difference': absolute_validation_difference,
    'Training seconds': training_seconds,
    'Prediction seconds': prediction_seconds,
}])
per_class = pd.DataFrame(final_report).T.loc[CLASS_ORDER, ['precision', 'recall', 'f1-score', 'support']]
display(final_summary.round(6))
display(per_class.round(6))

## 13. Show the fixed-order confusion matrix

**What:** visualize actual versus predicted classes. **Why:** the error pattern matters, especially missed Poor cases. **Expected:** one matrix for the frozen Decision Tree only. **Check:** there is no comparison with another model.

In [ ]:
ConfusionMatrixDisplay(final_matrix, display_labels=CLASS_ORDER).plot(
    cmap='Blues', colorbar=False, values_format='d'
)
plt.title('Frozen Decision Tree — final test')
plt.show()

## 14. Apply the criteria frozen before access

**What:** evaluate each predefined portfolio criterion without changing it. **Why:** post-result rule changes would invalidate the final test. **Expected:** pass/fail rows and one overall result. **Check:** each class has its own F1 check and the overall result passes only if every row passes.

In [ ]:
criterion_rows = [
    {'Criterion': 'Final-test Macro F1 >= 0.58', 'Observed': final_macro_f1, 'Pass': final_macro_f1 >= 0.58},
    {'Criterion': 'Final-test Poor recall >= 0.50', 'Observed': poor_recall, 'Pass': poor_recall >= 0.50},
]
for class_name in CLASS_ORDER:
    class_f1 = final_report[class_name]['f1-score']
    criterion_rows.append({'Criterion': f'{class_name} F1 >= 0.45', 'Observed': class_f1, 'Pass': class_f1 >= 0.45})
criterion_rows.append({
    'Criterion': 'Absolute validation-to-final Macro F1 difference <= 0.04',
    'Observed': absolute_validation_difference,
    'Pass': absolute_validation_difference <= 0.04,
})
criteria_results = pd.DataFrame(criterion_rows)
criteria_results['Result'] = np.where(criteria_results['Pass'], 'PASS', 'FAIL')
overall_portfolio_acceptance = bool(criteria_results['Pass'].all())
display(criteria_results[['Criterion', 'Observed', 'Result']].round({'Observed': 6}))
print(f"Overall educational portfolio acceptance: {'PASS' if overall_portfolio_acceptance else 'FAIL'}")

## 15. Close and lock the evaluation

**What:** verify the frozen classifier parameters and state the post-test boundary. **Why:** final-test results cannot justify another model, feature, threshold, or hyperparameter change. **Expected:** explicit safety statements. **Check:** no later training cell exists and no model or dataset is saved.

In [ ]:
classifier = frozen_pipeline.named_steps['classifier']
for parameter_name, frozen_value in FROZEN_MODEL_PARAMETERS.items():
    assert classifier.get_params()[parameter_name] == frozen_value
assert 'candidate_models' not in globals()
assert 'grid_search' not in globals()
assert 'randomized_search' not in globals()

print('Confirmed: exactly one frozen Decision Tree was evaluated on final test exactly once.')
print('Confirmed: no raw/transformed dataset or model was persisted.')
print('Locked: final-test results cannot be used for further model selection or tuning.')
print('This educational portfolio result does not establish production or lending readiness.')